# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm picking Lane 2: Refresh / Content Opportunity Scoring. I chose this over the other three because it gives me a concrete decision to optimize for from day one — which pages a reviewer should look at first — instead of an open-ended "find interesting patterns" question (Lane 1) or an unsupervised problem (Lane 3) where defending "why does this matter" is harder for a first project. The starter pipeline already builds this end to end (baseline score -> model -> ranked queue), which gives me something concrete to understand deeply before I try to improve on it.

In [7]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
# Quick sanity check: confirm the lane's core columns exist and aren't fully missing
print(df[['trend_direction', 'impressions_90d', 'days_since_last_update', 'avg_position']].isna().sum())

trend_direction           0
impressions_90d           0
days_since_last_update    0
avg_position              0
dtype: int64


## 2. The question: decision, action, cost of a wrong call

Decision: which content pages should a reviewer spend limited time on first, out of thousands.
Who acts: a content/SEO reviewer with a fixed weekly review capacity, not unlimited time. Action: they open the flagged page and decide to refresh, expand, or leave it.

Cost of a wrong call:
- False positive (flagged a page that didn't need review): wastes the reviewer's limited time on a page that wasn't actually a priority.
- False negative (missed a real declining page): a genuinely losing page keeps losing visibility/traffic with nobody looking at it, and the cost compounds the longer it's missed.

Since reviewer time is the scarce resource, this is fundamentally a ranking problem, not a "yes/no is this page bad" problem -- the order of the queue matters as much as who's on it.

In [9]:
# How big is the review queue, given limited reviewer capacity?
declining_with_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]

reviewer_capacity_per_week = 50
weeks_to_clear_backlog = len(declining_with_demand) / reviewer_capacity_per_week
print(f"Declining-with-demand pages: {len(declining_with_demand)}")
print(f"At {reviewer_capacity_per_week} pages/week, clearing this backlog would take ~{weeks_to_clear_backlog:.0f} weeks")
print("This is why ranking (who goes first) matters more than a flat yes/no flag.")

Declining-with-demand pages: 13152
At 50 pages/week, clearing this backlog would take ~263 weeks
This is why ranking (who goes first) matters more than a flat yes/no flag.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset (30,000 pages) back this lane:
- 43.8% of pages (13,152) are currently declining while still getting real search demand
  (impressions_90d >= 100) -- a large enough pool that prioritization genuinely matters; a reviewer can't just "do all of them."
- Only 0.1% of pages (17) are simply stale-but-visible (untouched 180+ days, still getting impressions >=500). That's a small number relative to the declining group above, which tells me "just refresh old pages" is not where the signal lives -- decline is happening for reasons other than staleness, which is worth digging into over the next 7 weeks rather than assuming.
- 23.6% of pages (7,076) sit in page-one-decay-risk territory (top-10 position, 180+ days old) -- a meaningful chunk of otherwise-successful content that could be quietly losing ground.

In [10]:
# How many pages look like real refresh candidates, using the lane guide's own baseline rules

total_pages = len(df)

stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
declining_with_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
page_one_decay = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['content_age_days'] >= 180)]

print(f"Total pages in starter dataset: {total_pages}")
print(f"Stale but visible pages (update >=180d, impressions >=500): {len(stale_visible)} ({len(stale_visible)/total_pages:.1%})")
print(f"Declining with real demand (trend=down, impressions >=100): {len(declining_with_demand)} ({len(declining_with_demand)/total_pages:.1%})")
print(f"Page-one decay risk (position 1-10, age >=180d): {len(page_one_decay)} ({len(page_one_decay)/total_pages:.1%})")

Total pages in starter dataset: 30000
Stale but visible pages (update >=180d, impressions >=500): 17 (0.1%)
Declining with real demand (trend=down, impressions >=100): 13152 (43.8%)
Page-one decay risk (position 1-10, age >=180d): 7076 (23.6%)


## 4. Careful words: what I can and can't claim

What I can claim: observed patterns in this anonymized 90-day snapshot -- which pages are declining, how common it is, which signals correlate with it. My rankings are decision-support: they help a human reviewer prioritize, not decide automatically.

What I can never claim: that any signal here causes a decline or that fixing a page will cause recovery (that needs a real experiment, not this data). I also can't claim anything about Google's actual ranking algorithm -- I only have FlyRank's own observed search and engagement metrics, not insight into how Google decides rankings.

In [11]:
# Example of a claim I explicitly can't make: AI-referral visibility
ai_present = (df['ai_sessions_90d'] > 0).sum()
print(f"Pages with any AI-referral sessions: {ai_present} of {total_pages} ({ai_present/total_pages:.1%})")
print("Too sparse to claim anything meaningful about AI search visibility from this alone.")

Pages with any AI-referral sessions: 1930 of 30000 (6.4%)
Too sparse to claim anything meaningful about AI search visibility from this alone.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.